In [1]:
# ==========================================
# CELLA 1: SETUP E ONE-HOT ENCODING 
# ==========================================

import numpy as np
import chess

# Mappa statica globale: leggerla da qui è istantaneo per il processore
# Bianco: 0-5 | Nero: 6-11
PIECE_MAP = {
    'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
    'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11
}

def fen_to_768_fast(fen):
    """
    Versione ultra-veloce: non crea l'oggetto chess.Board, ma legge direttamente 
    la stringa di testo. Indispensabile per processare milioni di righe.
    """
    vector = np.zeros(768, dtype=np.float32)
    
    # Prendiamo solo la prima parte del FEN (la scacchiera), ignorando i turni
    board_part = fen.split(' ')[0]
    
    # Il FEN parte dalla riga 8 (indice 7) e scende
    rank = 7
    file = 0
    
    for char in board_part:
        if char == '/':
            rank -= 1    # Scendiamo di una riga
            file = 0     # Torniamo alla colonna 'a'
        elif char.isdigit():
            file += int(char) # Saltiamo le case vuote
        else:
            # Abbiamo trovato un pezzo!
            idx = PIECE_MAP[char]
            square = rank * 8 + file
            flat_index = (idx * 64) + square
            
            vector[flat_index] = 1.0
            file += 1
            
    return vector

print("✅ Cella 1 caricata: Funzione FEN ultra-veloce pronta.")

✅ Cella 1 caricata: Funzione FEN ultra-veloce pronta.


In [3]:
# ==========================================
# CELLA 2: ESTRAZIONE DELLE 13 FEATURE (Notebook 1)
# ==========================================
import numpy as np
import chess

def extract_13_features(fen):
    """Calcola le 13 feature euristiche (inclusi Scacchi e Torri comunicanti)."""
    board = chess.Board(fen)
    features = np.zeros(13, dtype=np.float32) 
    
    valori = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    mat_W = mat_B = center_W = center_B = dev_W = dev_B = harmony_W = harmony_B = 0
    case_centrali = [chess.E4, chess.D4, chess.E5, chess.D5]
    
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if not piece: continue
        color = piece.color
        ptype = piece.piece_type
        rank = chess.square_rank(sq)
        
        if ptype != chess.KING:
            if color == chess.WHITE: mat_W += valori[ptype]
            else: mat_B += valori[ptype]
        for c_sq in case_centrali:
            if board.is_attacked_by(color, c_sq):
                if color == chess.WHITE: center_W += 1
                else: center_B += 1
        if ptype in [chess.KNIGHT, chess.BISHOP]:
            if color == chess.WHITE and rank == 0: dev_W -= 1
            if color == chess.BLACK and rank == 7: dev_B -= 1
        if len(board.attackers(color, sq)) > 0:
            if color == chess.WHITE: harmony_W += 1
            else: harmony_B += 1

    features[0] = mat_W - mat_B
    features[1] = (0.5 if len(board.pieces(chess.BISHOP, chess.WHITE)) >= 2 else 0) - (0.5 if len(board.pieces(chess.BISHOP, chess.BLACK)) >= 2 else 0)
    features[2] = ((len(board.pieces(chess.KNIGHT, chess.WHITE)) + len(board.pieces(chess.BISHOP, chess.WHITE))) - (len(board.pieces(chess.ROOK, chess.WHITE)) + len(board.pieces(chess.QUEEN, chess.WHITE)))) - ((len(board.pieces(chess.KNIGHT, chess.BLACK)) + len(board.pieces(chess.BISHOP, chess.BLACK))) - (len(board.pieces(chess.ROOK, chess.BLACK)) + len(board.pieces(chess.QUEEN, chess.BLACK))))
    
    board.turn = chess.WHITE; mob_W = board.legal_moves.count()
    board.turn = chess.BLACK; mob_B = board.legal_moves.count()
    features[3] = mob_W - mob_B
    features[4] = center_W - center_B
    features[5] = dev_W - dev_B
    features[6] = 0.0 
    
    f_W = [chess.square_file(sq) for sq in board.pieces(chess.PAWN, chess.WHITE)]
    f_B = [chess.square_file(sq) for sq in board.pieces(chess.PAWN, chess.BLACK)]
    features[7] = -((len(f_W) - len(set(f_W))) - (len(f_B) - len(set(f_B))))
    features[8] = 0.0 
    
    king_W = board.king(chess.WHITE)
    king_B = board.king(chess.BLACK)
    features[9] = (-sum(len(board.attackers(chess.BLACK, adj)) for adj in chess.SQUARES if king_W and chess.square_distance(king_W, adj) <= 1)) - (-sum(len(board.attackers(chess.WHITE, adj)) for adj in chess.SQUARES if king_B and chess.square_distance(king_B, adj) <= 1))
    features[10] = harmony_W - harmony_B
    
    # 12. Torri Comunicanti
    conn_W = conn_B = 0
    rooks_W = list(board.pieces(chess.ROOK, chess.WHITE))
    if len(rooks_W) >= 2 and set(board.attackers(chess.WHITE, rooks_W[0])) & set(rooks_W): conn_W = 1
    rooks_B = list(board.pieces(chess.ROOK, chess.BLACK))
    if len(rooks_B) >= 2 and set(board.attackers(chess.BLACK, rooks_B[0])) & set(rooks_B): conn_B = 1
    features[11] = conn_W - conn_B

    # 13. Possibili Scacchi
    original_turn = board.turn 
    board.turn = chess.WHITE
    features[12] = sum(1 for m in board.legal_moves if board.gives_check(m))
    board.turn = chess.BLACK
    features[12] -= sum(1 for m in board.legal_moves if board.gives_check(m))
    board.turn = original_turn 

    return features

In [4]:
# ==========================================
# CELLA 3: ELABORAZIONE MASSIVA (Notebook 1)
# ==========================================
import pandas as pd
from tqdm import tqdm

print("Caricamento del dataset in corso...")
df = pd.read_csv('./Data/fen_analysis.csv') 
N_righe = len(df)

evals = df['Analysis'].astype(str).str.strip()
mask_no_mate = ~evals.str.contains('#')
valutazioni_normali = evals[mask_no_mate].astype(float)
VERO_MAX = valutazioni_normali.max()
VERO_MIN = valutazioni_normali.min()

def clean_evaluation_rigorosa(val):
    val_str = str(val).strip()
    if '#' in val_str:
        if '+' in val_str or (len(val_str) > 1 and val_str[1] != '-'): return float(VERO_MAX) 
        else: return float(VERO_MIN)
    try: return float(val_str)
    except ValueError: return 0.0

print("Allocazione della memoria...")
X_768 = np.zeros((N_righe, 768), dtype=np.float32)
X_13  = np.zeros((N_righe, 13), dtype=np.float32) # AGGIORNATO A 13
Y     = np.zeros(N_righe, dtype=np.float32)

for i, row in tqdm(df.iterrows(), total=N_righe, desc="Processando FEN"):
    fen = row['FEN']
    
    X_768[i] = fen_to_768_fast(fen) 
    X_13[i]  = extract_13_features(fen) # AGGIORNATO ALLA NUOVA FUNZIONE
    Y[i]     = clean_evaluation_rigorosa(row['Analysis'])

nome_file_salvataggio = 'dataset_elaborato.npz'
print(f"\nSalvataggio dei dati in {nome_file_salvataggio}...")
np.savez_compressed(nome_file_salvataggio, X_768=X_768, X_13=X_13, Y=Y) # SALVA LA NUOVA MATRICE
print("✅ Finito! Dati salvati con successo.")

Caricamento del dataset in corso...
Allocazione della memoria...


Processando FEN: 100%|██████████| 336903/336903 [04:54<00:00, 1142.95it/s]



Salvataggio dei dati in dataset_elaborato.npz...
✅ Finito! Dati salvati con successo.
